# NN Architecture 2B: 1D Convolutional Neural Network (CNN)

**Reference**: "Binary Case using Deep Learning" (project paper)

**Approach**: Learn local patterns in signal sequences (BPSK + TAG) through convolutional filters

**Rationale**: Convolutional layers with 1D kernels excel at:
- Detecting local correlations in signal sequences
- Learning shift-invariant features
- Reducing parameters vs. fully-connected layers
- Handling variable-length inputs (with padding)

**Architecture**:
```
Input: [τ_signal, h_magnitude, SNR_local, energy] (4D, reshaped as 1D sequence)
    ↓
Conv1D(32, kernel=5, padding='same', ReLU)
    ↓
BatchNorm → MaxPool(2, stride=2)
    ↓
Conv1D(64, kernel=3, padding='same', ReLU)
    ↓
BatchNorm → MaxPool(2, stride=2) → Flatten
    ↓
Dense(128, ReLU, Dropout=0.4)
    ↓
Dense(64, ReLU, Dropout=0.3)
    ↓
Dense(1, Sigmoid) → Binary output
```

**Framework**: PyTorch (better for signal processing + flexibility)

In [ ]:
# ==============================================================================
# 1. IMPORTS & SETUP
# ==============================================================================

import numpy as np
import matplotlib.pyplot as plt
import h5py
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    roc_auc_score, confusion_matrix, roc_curve
)
import warnings
warnings.filterwarnings('ignore')

# GPU setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Load dataset
dataset_path = "dataset_nn_100k.h5"
with h5py.File(dataset_path, 'r') as f:
    X_train = f['X_train'][:]
    y_train = f['y_train'][:]
    X_val = f['X_val'][:]
    y_val = f['y_val'][:]
    X_test = f['X_test'][:]
    y_test = f['y_test'][:]

print(f"✓ Dataset loaded: X_train={X_train.shape}, y_train={y_train.shape}")

In [ ]:
# ==============================================================================
# 2. BUILD CNN ARCHITECTURE
# ==============================================================================

class CNN_SignalProcessing(nn.Module):
    """1D CNN for signal processing (Binary Case using Deep Learning)"""
    
    def __init__(self, input_channels=1, dropout_p=0.3):
        super(CNN_SignalProcessing, self).__init__()
        
        # Conv block 1
        self.conv1 = nn.Conv1d(input_channels, 32, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm1d(32)
        self.pool1 = nn.MaxPool1d(2, stride=2)
        self.dropout1 = nn.Dropout(dropout_p)
        
        # Conv block 2
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(64)
        self.pool2 = nn.MaxPool1d(2, stride=2)
        self.dropout2 = nn.Dropout(dropout_p)
        
        # FC layers
        # After 2 maxpools, input_dim=4 → 4/4 ≈ 1, but we keep it larger for flexibility
        self.fc1 = nn.Linear(64 * 1, 128)  # 64 channels * reduced spatial dims
        self.dropout3 = nn.Dropout(0.4)
        
        self.fc2 = nn.Linear(128, 64)
        self.dropout4 = nn.Dropout(0.3)
        
        self.fc3 = nn.Linear(64, 1)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        # Conv block 1
        x = self.conv1(x)
        x = torch.relu(self.bn1(x))
        x = self.pool1(x)
        x = self.dropout1(x)
        
        # Conv block 2
        x = self.conv2(x)
        x = torch.relu(self.bn2(x))
        x = self.pool2(x)
        x = self.dropout2(x)
        
        # Flatten
        x = x.view(x.size(0), -1)
        
        # FC layers
        x = torch.relu(self.fc1(x))
        x = self.dropout3(x)
        
        x = torch.relu(self.fc2(x))
        x = self.dropout4(x)
        
        x = self.fc3(x)
        x = self.sigmoid(x)
        
        return x

# Create model
model = CNN_SignalProcessing(input_channels=1, dropout_p=0.3)
model = model.to(device)

print("✓ CNN Model created:")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(model)

In [ ]:
# ==============================================================================
# 3. PREPARE DATA & TRAINING SETUP
# ==============================================================================

# Reshape for CNN: (N, 4) → (N, 1, 4) for Conv1D
X_train_cnn = torch.FloatTensor(X_train).unsqueeze(1)  # (N, 1, 4)
y_train_cnn = torch.FloatTensor(y_train).unsqueeze(1)  # (N, 1)

X_val_cnn = torch.FloatTensor(X_val).unsqueeze(1)
y_val_cnn = torch.FloatTensor(y_val).unsqueeze(1)

X_test_cnn = torch.FloatTensor(X_test).unsqueeze(1)
y_test_cnn = torch.FloatTensor(y_test).unsqueeze(1)

# Create DataLoader
batch_size = 256
train_dataset = TensorDataset(X_train_cnn, y_train_cnn)
val_dataset = TensorDataset(X_val_cnn, y_val_cnn)
test_dataset = TensorDataset(X_test_cnn, y_test_cnn)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

# Loss & Optimizer (with class weights)
class_weights = torch.tensor(
    [np.sum(y_train == 1) / np.sum(y_train == 0), 1.0],
    device=device
)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-6
)

print(f"✓ Data loaders created: batch_size={batch_size}")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")
print(f"  Test batches: {len(test_loader)}")

In [ ]:
# ==============================================================================
# 4. TRAINING LOOP
# ==============================================================================

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * X_batch.size(0)
    
    return total_loss / len(loader.dataset)

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            
            total_loss += loss.item() * X_batch.size(0)
            all_preds.extend(y_pred.cpu().numpy().flatten())
            all_targets.extend(y_batch.cpu().numpy().flatten())
    
    return total_loss / len(loader.dataset), np.array(all_preds), np.array(all_targets)

# Training
num_epochs = 100
patience = 10
best_val_loss = float('inf')
patience_counter = 0
history = {'train_loss': [], 'val_loss': []}

print("Training CNN model...")
for epoch in range(num_epochs):
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, _, _ = evaluate(model, val_loader, criterion, device)
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    
    scheduler.step(val_loss)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), 'model_cnn_best.pth')
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

print("✓ Training completed!")

In [ ]:
# ==============================================================================
# 5. EVALUATION & METRICS
# ==============================================================================

# Load best model
model.load_state_dict(torch.load('model_cnn_best.pth'))

# Get predictions on all sets
_, y_train_pred_proba, y_train_targets = evaluate(model, train_loader, criterion, device)
_, y_val_pred_proba, y_val_targets = evaluate(model, val_loader, criterion, device)
_, y_test_pred_proba, y_test_targets = evaluate(model, test_loader, criterion, device)

# Binary predictions
y_train_pred = (y_train_pred_proba > 0.5).astype(int)
y_val_pred = (y_val_pred_proba > 0.5).astype(int)
y_test_pred = (y_test_pred_proba > 0.5).astype(int)

# Calculate metrics
def calculate_metrics(y_true, y_pred, y_pred_proba, name):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    auc = roc_auc_score(y_true, y_pred_proba)
    
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    
    print(f"\n{name}: Acc={acc:.4f}, Prec={prec:.4f}, Rec={rec:.4f}, F1={f1:.4f}, AUC={auc:.4f}")
    print(f"  FNR={fnr:.6f}, FPR={fpr:.6f}")
    
    return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1, 
            'auc': auc, 'fnr': fnr, 'fpr': fpr}

metrics_train_cnn = calculate_metrics(y_train_targets, y_train_pred, y_train_pred_proba, "Train")
metrics_val_cnn = calculate_metrics(y_val_targets, y_val_pred, y_val_pred_proba, "Val")
metrics_test_cnn = calculate_metrics(y_test_targets, y_test_pred, y_test_pred_proba, "Test")

# Plots
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss curve
axes[0].plot(history['train_loss'], label='Train', linewidth=2)
axes[0].plot(history['val_loss'], label='Val', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('CNN Training History')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# ROC Curve
fpr_test, tpr_test, _ = roc_curve(y_test_targets, y_test_pred_proba)
axes[1].plot(fpr_test, tpr_test, linewidth=2, label=f"AUC={metrics_test_cnn['auc']:.4f}")
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[1].set_xlabel('FPR')
axes[1].set_ylabel('TPR')
axes[1].set_title('ROC Curve (Test)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Prediction distribution
axes[2].hist(y_test_pred_proba[y_test_targets==0], alpha=0.6, label='Fraudulent', bins=40)
axes[2].hist(y_test_pred_proba[y_test_targets==1], alpha=0.6, label='Authentic', bins=40)
axes[2].axvline(0.5, color='red', linestyle='--', linewidth=2)
axes[2].set_xlabel('Predicted Probability')
axes[2].set_ylabel('Count')
axes[2].set_title('Prediction Distribution (Test)')
axes[2].legend()

plt.tight_layout()
plt.savefig('results_cnn.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Results saved to 'results_cnn.png'")